In [0]:
from pyspark.sql.functions import col, explode

print("Iniciando a Camada Prata: Desaninhamento Estrutural (Autores)...")

# Leitura da Camada Bronze (fonte imutável)
df_bronze = spark.table("bronze_openalex")

# Desaninhamento do array de autores (flattening estrutural sem aplicar regras de negócio da Ouro)
df_autores_achatado = df_bronze.select(
    col("id").alias("work_id"),
    explode(col("authorships")).alias("authorship")
)

# Extração de atributos do dicionário interno, padronização e purificação (remoção de nulos/duplicados)
df_autores_limpo = df_autores_achatado.select(
    col("work_id"),
    col("authorship.author.id").alias("author_id"),
    col("authorship.author.display_name").alias("author_name"),
    col("authorship.author_position").alias("author_position")
).filter(col("author_id").isNotNull()).dropDuplicates()

print(f"Total de registros desaninhados e limpos na Prata: {df_autores_limpo.count()}")

# Persistência física no formato colunar (Delta Lake)
tabela_prata_autores = "silver_authors_flattened"
df_autores_limpo.write.format("delta").mode("overwrite").saveAsTable(tabela_prata_autores)

print(f"\nTabela Delta '{tabela_prata_autores}' gerada com sucesso! O dado estrutural está pronto para a Camada Ouro.")
display(df_autores_limpo.limit(10))